# FedGNN-Halo passo a passo

Este notebook mostra, em um grafo pequeno, o que o FedGNN-Halo faz e o que ele preserva. Faz parte do código da dissertação de mestrado *Aprendizado Federado em Grafos Espaço-Temporais Particionados: Um Estudo de Caso em Mobilidade Urbana* (PGCOMP/UFBA, [LabIA](https://labia.ufba.br/)).

Não há treinamento: todas as comparações usam **os mesmos parâmetros** em uma única camada LEConv. O objetivo é verificar a operação espacial, não o desempenho preditivo.

1. Um grafo com dois clientes e as arestas cortadas entre eles.
2. O cliente sem halo: as mensagens dos vizinhos externos deixam de chegar.
3. O agregado de cada nó com entrada externa e o nó virtual que o representa.
4. A saída da LEConv no cliente comparada à do grafo completo.

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import networkx as nx

from models import ModelSpec, build_model
from federation.halo import aggregate_halo

torch.manual_seed(0)
rng = np.random.default_rng(0)

## 1. Grafo e partição

Oito nós, divididos em dois clientes. `adj[u, v]` é o peso da aresta de `u` para `v`. Todo nó tem um autolaço, como nos grafos usados nos experimentos. As arestas tracejadas ligam clientes diferentes e são cortadas no treinamento local.

In [ ]:
N = 8
edges = [(0, 1, 1.0), (0, 2, 0.5), (1, 2, 2.0), (2, 3, 1.0),   # cliente 0
         (4, 5, 1.0), (4, 7, 0.5), (5, 6, 2.0), (6, 7, 1.0),   # cliente 1
         (2, 5, 1.0), (3, 4, 1.5), (3, 5, 1.0)]                # entre clientes
adj = np.eye(N, dtype=np.float32)
for u, v, w in edges:
    adj[u, v] = adj[v, u] = w
client_of = np.array([0, 0, 0, 0, 1, 1, 1, 1])

G = nx.Graph()
G.add_weighted_edges_from(edges)
pos = {0: (0, 1), 1: (0, -1), 2: (1, 0), 3: (2, 0.8), 4: (3, 0.8), 5: (3, -0.8), 6: (4, -1), 7: (4, 1)}
cut = [(u, v) for u, v, _ in edges if client_of[u] != client_of[v]]
internal = [(u, v) for u, v, _ in edges if client_of[u] == client_of[v]]

fig, ax = plt.subplots(figsize=(6, 3))
nx.draw_networkx_edges(G, pos, edgelist=internal, width=1.5, ax=ax)
nx.draw_networkx_edges(G, pos, edgelist=cut, width=1.5, style="dashed", edge_color="#b0b0b0", ax=ax)
nx.draw_networkx_nodes(G, pos, node_color=["#2a78d6" if c == 0 else "#c0612b" for c in client_of],
                       node_size=420, ax=ax)
nx.draw_networkx_labels(G, pos, font_color="white", ax=ax)
nx.draw_networkx_edge_labels(G, pos, edge_labels={(u, v): f"{w:g}" for u, v, w in edges}, font_size=8, ax=ax)
ax.set_title("Cliente 0 (azul) e cliente 1 (laranja); tracejado: arestas cortadas", fontsize=9)
ax.axis("off")
plt.show()

## 2. Modelo de referência no grafo completo

A LEConv normaliza os pesos de entrada por nó de destino. Construímos o modelo no grafo completo e guardamos seus parâmetros; os modelos dos clientes recebem exatamente esses parâmetros.

A entrada é uma janela de seis instantes com valores já padronizados e máscara de observação (aqui, tudo observado).

In [ ]:
W, H, T = 6, 2, 40
series = rng.normal(size=(T, N, 1)).astype(np.float32)    # valores padronizados
mask = np.ones_like(series)

def edges_of(matrix):
    src, dst = np.nonzero(matrix)
    return torch.tensor(np.stack([src, dst]), dtype=torch.long), torch.tensor(matrix[src, dst], dtype=torch.float32)

def leconv(edge_index, edge_weight, nodes):
    spec = ModelSpec(nodes=nodes, input_steps=W, horizon=H, n_features=1,
                     edge_index=edge_index, edge_weight=edge_weight)
    return build_model("leconv", spec, hidden_size=8).eval()

central = leconv(*edges_of(adj), N)
state = central.state_dict()

x = torch.from_numpy(series[:W]).unsqueeze(0)              # (1, W, N, 1)
m = torch.from_numpy(mask[:W]).unsqueeze(0)
with torch.no_grad():
    reference = central(x, m)[0, :, :, 0]                  # (H, N)

## 3. Cliente 0 sem halo

O subgrafo do cliente mantém só as arestas internas. Os nós 2 e 3 recebiam mensagens do cliente 1 e passam a normalizar os pesos apenas sobre os vizinhos que restaram.

In [ ]:
nodes = np.flatnonzero(client_of == 0)
sub = adj[np.ix_(nodes, nodes)]
none_model = leconv(*edges_of(sub), len(nodes))
none_model.load_state_dict(state)
with torch.no_grad():
    out_none = none_model(x[:, :, nodes], m[:, :, nodes])[0, :, :, 0]

## 4. Agregados e nós virtuais

Para cada nó local $v$ com entrada externa, `aggregate_halo` calcula

$$a_v(t) = \frac{\sum_{u \notin \text{cliente},\, u \to v} w_{uv}\, x_u(t)}{W_v}, \qquad W_v = \sum_{u \notin \text{cliente},\, u \to v} w_{uv},$$

e o mesmo para a máscara. O cliente recebe um nó virtual por nó com entrada externa, ligado a ele por uma aresta de peso $W_v$, com autolaço. A soma dos pesos de entrada de $v$ volta a ser a do grafo completo.

In [ ]:
halo = aggregate_halo(adj, nodes, series, mask)
pd.DataFrame({
    "nó local": nodes[halo.boundary_local],
    "vizinhos externos": halo.n_external,
    "W_v": halo.weight,
})

In [ ]:
# O nó 3 recebe dos nós 4 (peso 1,5) e 5 (peso 1,0): o agregado é a média ponderada das duas séries.
i3 = list(nodes[halo.boundary_local]).index(3)
expected = (1.5 * series[:, 4] + 1.0 * series[:, 5]) / 2.5
print("nó 3, agregado = (1,5·x4 + 1,0·x5) / 2,5:", np.allclose(halo.values[:, i3], expected, atol=1e-6))

# O nó 2 tem um único vizinho externo (nó 5): o agregado é a própria série desse vizinho.
i2 = list(nodes[halo.boundary_local]).index(2)
print("nó 2, agregado = x5:", np.allclose(halo.values[:, i2], series[:, 5]))

O segundo caso mostra um limite do método: quando há um único vizinho externo, o agregado reproduz a série padronizada desse vizinho. Na dissertação, isso ocorre na maioria dos nós virtuais do SUNT.

In [ ]:
n, h = len(nodes), halo.size
virtual = n + np.arange(h)
ei, ew = edges_of(sub)
extra_index = torch.tensor(np.stack([np.r_[virtual, virtual], np.r_[halo.boundary_local, virtual]]), dtype=torch.long)
extra_weight = torch.tensor(np.r_[halo.weight, np.ones(h, dtype=np.float32)])
halo_model = leconv(torch.cat([ei, extra_index], 1), torch.cat([ew, extra_weight]), n + h)
halo_model.load_state_dict(state)

x_halo = torch.cat([x[:, :, nodes], torch.from_numpy(halo.values[:W]).unsqueeze(0)], dim=2)
m_halo = torch.cat([m[:, :, nodes], torch.from_numpy(halo.mask[:W]).unsqueeze(0)], dim=2)
with torch.no_grad():
    out_halo = halo_model(x_halo, m_halo)[0, :, :n, 0]   # só os nós locais

## 5. Comparação com o grafo completo

Diferença absoluta máxima, ao longo do horizonte, entre a saída de cada nó do cliente 0 e a saída do mesmo nó no grafo completo.

In [ ]:
ref = reference[:, nodes]
table = pd.DataFrame({
    "nó": nodes,
    "entrada externa": np.isin(np.arange(n), halo.boundary_local),
    "sem halo": (out_none - ref).abs().max(0).values.numpy(),
    "com halo": (out_halo - ref).abs().max(0).values.numpy(),
})
table.style.format({"sem halo": "{:.2e}", "com halo": "{:.2e}"})

Os nós 0 e 1 não têm vizinhos em outro cliente e coincidem com o grafo completo nas duas condições. Nos nós 2 e 3, a saída sem halo difere; com halo, a diferença fica no nível do arredondamento em ponto flutuante.

**O que isso mostra e o que não mostra:**

- a igualdade vale para **uma camada LEConv**, com os **mesmos parâmetros** e a mesma normalização. Com mais camadas, seriam necessários agregados de vizinhos mais distantes;
- não implica o mesmo modelo depois do treinamento federado, que muda os parâmetros compartilhados;
- na simulação, os agregados são preparados com acesso a todas as séries e não oferecem, por si, proteção dos dados.

No treinamento, o método `halo` repete esta verificação no início de cada execução e registra o erro máximo no log (`exatidão vs centralizado`). Os notebooks [01](01_quickstart.ipynb) e [02](02_methods_and_partitions.ipynb) mostram o treinamento com a API.